# Features:

RowNumber: The sequential number assigned to each row in the dataset.

CustomerId: A unique identifier for each customer.

Surname: The surname of the customer.

CreditScore: The credit score of the customer.

Geography: The geographical location of the customer (e.g., country or region).

Gender: The gender of the customer.

Age: The age of the customer.

Tenure: The number of years the customer has been with the bank.

Balance: The account balance of the customer.

NumOfProducts: The number of bank products the customer has.

HasCrCard: Indicates whether the customer has a credit card (binary: yes/no).

IsActiveMember: Indicates whether the customer is an active member (binary: yes/no).

EstimatedSalary: The estimated salary of the customer.

Exited: Indicates whether the customer has exited the bank (binary: yes/no).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (roc_auc_score, recall_score,accuracy_score,
                                     precision_score, classification_report)
from sklearn.ensemble import RandomForestClassifier

In [3]:
df=pd.read_csv("data\\Churn_Modelling.csv")

In [ ]:
df.info()

<bound method DataFrame.info of       RowNumber  CustomerId    Surname  CreditScore Geography  Gender  Age  \
0             1    15634602   Hargrave          619    France  Female   42   
1             2    15647311       Hill          608     Spain  Female   41   
2             3    15619304       Onio          502    France  Female   42   
3             4    15701354       Boni          699    France  Female   39   
4             5    15737888   Mitchell          850     Spain  Female   43   
...         ...         ...        ...          ...       ...     ...  ...   
9995       9996    15606229   Obijiaku          771    France    Male   39   
9996       9997    15569892  Johnstone          516    France    Male   35   
9997       9998    15584532        Liu          709    France  Female   36   
9998       9999    15682355  Sabbatini          772   Germany    Male   42   
9999      10000    15628319     Walker          792    France  Female   28   

      Tenure    Balance  NumOfP

In [5]:
class BankFeatureEngineering(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self

    def transform(self,X):
        df= X.copy()

        def categorize_tenure(tenure):
            if pd.isna(tenure):
                return 'Unknown'
            elif tenure <= 2:
                return 'New (<2 yrs)'
            elif tenure <= 5:
                return 'Short-term (2-5 yrs)'
            elif tenure <= 8:
                return 'Medium-term (6-8 yrs)'
            else:
                return 'Long-term (>8 yrs)'
        
        def categorize_balance(balance):
            if pd.isna(balance):
                return 'Unknown'
            elif balance == 0:
                return 'Zero Balance'
            elif balance < 50000:
                return 'Low (<50K)'
            elif balance < 150000:
                return 'Medium (50K-150K)'
            else:
                return 'High (>150K)'
        
        
        def age_group(CreditScore):
            if pd.isna(CreditScore):
                return 'Unknown'
            elif CreditScore<400:
                return 'Low Credit'
            elif CreditScore<550:
                return 'Okay Credit'
            elif CreditScore<650:
                return 'Good credit'
            else:
                return 'Excellent'
        
        
        
        def age_group(age):
            if pd.isna(age):
                return 'Unknown'
            elif age<30:
                return 'Young'
            elif age<45:
                return 'Adult'
            elif age<60:
                return 'Middle-Aged'
            else:
                return 'Senior'
        
        df['TenureGroup'] = df['Tenure'].apply(categorize_tenure)
        df['CreditScore Range']= df['CreditScore'].apply(age_group)
        df['BalanceGroup'] = df['Balance'].apply(categorize_balance)
        df['AgeGroup']= df['Age'].apply(age_group)

        cols_to_drop= ['RowNumber', 'CustomerId', 'Surname']
        df= df.drop(columns=[col for col in cols_to_drop if col in df.columns])
        return df

numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
categorical_cols = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember', 'AgeGroup', 'BalanceGroup', 'CreditScore Range']

# standardising and converting cat to nums
numeric_transformer= Pipeline(steps=[('scaler',StandardScaler())])
categorical_transformer= Pipeline(steps=[('onehot',OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor= ColumnTransformer(
    transformers=[
        ('num',numeric_transformer,numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)


full_preprocessing_pipeline = Pipeline(steps=[
    ('feature_engineering', BankFeatureEngineering()),
    ('preprocessor', preprocessor)
])

In [6]:
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y
)

In [7]:
# Fit & transform on training data
X_train_processed = full_preprocessing_pipeline.fit_transform(X_train)
X_test_processed = full_preprocessing_pipeline.transform(X_test)

model=RandomForestClassifier(random_state=42)
model.fit(X_train_processed,y_train)
y_pred = model.predict(X_test_processed)

# Probabilities
rf_prob = model.predict_proba(X_test_processed)[:, 1]

# Custom threshold

threshold = 0.35

rf_pred= (rf_prob >= threshold).astype(int)

 
print(f"AUC-ROC : {roc_auc_score(y_test, rf_prob):.4f}")
print(f"Recall  : {recall_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=["Not Churned", "Churned"]))




AUC-ROC : 0.8458
Recall  : 0.6192
Precision: 0.6029

Classification Report:
              precision    recall  f1-score   support

 Not Churned       0.90      0.90      0.90      1593
     Churned       0.60      0.62      0.61       407

    accuracy                           0.84      2000
   macro avg       0.75      0.76      0.75      2000
weighted avg       0.84      0.84      0.84      2000



In [8]:

new_customers = pd.DataFrame({
    'CreditScore':[850],
    'Geography':'Germany',
    'Gender':'Male',
    'Age':45,
    'Tenure':5,
    'Balance':100000,
    'NumOfProducts':0,
    'HasCrCard':1,
    'IsActiveMember':0,
    'EstimatedSalary': 60000    
},index=[0])

print(new_customers)

   CreditScore Geography Gender  Age  Tenure  Balance  NumOfProducts  \
0          850   Germany   Male   45       5   100000              0   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               0            60000  


In [9]:
# Transform new test data prior to model inference
X_new_processed = full_preprocessing_pipeline.transform(new_customers)

# Pass directly to your trained model
predictions = model.predict(X_new_processed)

print ('1=churn,0=not churn')
print("The customer:",int(predictions))

1=churn,0=not churn
The customer: 1


C:\Users\prasa\AppData\Local\Temp\ipykernel_32232\612140917.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("The customer:",int(predictions))


In [12]:
print("Predictions for new customers:", int(predictions))

Predictions for new customers: 1


C:\Users\prasa\AppData\Local\Temp\ipykernel_32232\2264547045.py:1: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("Predictions for new customers:", int(predictions))
